# Wan 2.2 Animate — Character Replacement (Free T4)

Replace photorealistic characters in EP02 clips with 2D cartoon versions.
Runs on free Kaggle T4 (16GB VRAM) or Colab T4.

**Inputs you need:**
- Driving video: the EP02 clip with the photorealistic character (upload below)
- Character image: 2D cartoon reference of the same character (upload below)

**Output:** Same video, same motion, same background — but the character is now 2D.

## Step 1: Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Install ComfyUI + WanVideoWrapper

In [ ]:
# Clone ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt

# Custom nodes for Wan Animate
%cd /kaggle/working/ComfyUI/custom_nodes
!git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git
!git clone https://github.com/kijai/ComfyUI-KJNodes.git
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
!git clone https://github.com/Fannovel16/comfyui_controlnet_aux.git
!git clone https://github.com/city96/ComfyUI-GGUF.git
%cd /kaggle/working/ComfyUI/custom_nodes/ComfyUI-WanVideoWrapper
!pip install -q -r requirements.txt

## Step 3: Download models (GGUF quantized for T4)

In [ ]:
%cd /kaggle/working/ComfyUI
!mkdir -p models/diffusion_models models/vae models/text_encoders models/clip_vision models/detection

# Main model: Wan 2.2 Animate 14B GGUF Q3_K_M (~5GB, fits T4)
!wget -q --show-progress -O models/diffusion_models/Wan2.2-Animate-14B-Q3_K_M.gguf \
  https://huggingface.co/QuantStack/Wan2.2-Animate-14B-GGUF/resolve/main/Wan2.2-Animate-14B-Q3_K_M.gguf

# VAE
!wget -q --show-progress -O models/vae/wan_2.1_vae.safetensors \
  https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_1_VAE_bf16.safetensors

# Text encoder
!wget -q --show-progress -O models/text_encoders/umt5_xxl_fp16.safetensors \
  https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp16.safetensors

# Detection models (pose extraction)
!wget -q -O models/detection/yolov10m.onnx \
  https://huggingface.co/Wan-AI/Wan2.2-Animate-14B/resolve/main/process_checkpoint/det/yolov10m.onnx
!wget -q -O models/detection/vitpose-l-wholebody.onnx \
  https://huggingface.co/JunkyByte/easy_ViTPose/resolve/main/onnx/wholebody/vitpose-l-wholebody.onnx

print("Models downloaded. Check sizes:")
!du -sh models/diffusion_models/ models/vae/ models/text_encoders/ models/detection/

## Step 4: Upload your files

Upload:
1. `driving.mp4` — the EP02 clip (photorealistic character)
2. `character.png` — 2D cartoon reference image of the character

In [ ]:
# KAGGLE: Upload your files using the notebook's file upload UI
# (click the folder icon on the left → Upload)
# Then run this cell to organize them
import os, shutil, glob
os.makedirs("/kaggle/working/inputs", exist_ok=True)
%cd /kaggle/working/inputs
# Find uploaded files (Kaggle puts them in /kaggle/input/ or you upload via UI)
for search_dir in ["/kaggle/input", "."]:
    for f in glob.glob(f"{search_dir}/*"):
        fname = os.path.basename(f)
        if fname.endswith(('.mp4', '.mov', '.avi')) and not os.path.exists("driving.mp4"):
            shutil.copy(f, "driving.mp4")
            print(f"Driving video: {fname} -> driving.mp4")
        elif fname.endswith(('.png', '.jpg', '.jpeg')) and not os.path.exists("character.png"):
            shutil.copy(f, "character.png")
            print(f"Character image: {fname} -> character.png")
print("\nFiles ready. If empty, upload via the folder icon on the left sidebar.")
print(f"Contents: {os.listdir('.')}")


## Step 5: Start ComfyUI (background)

In [ ]:
import subprocess, time
proc = subprocess.Popen(
    ["python", "main.py", "--port", "8188"],
    cwd="/kaggle/working/ComfyUI",
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(15)
print("ComfyUI starting on port 8188...")
# For Kaggle: use cloudflared or ngrok to expose the port
!curl -s http://127.0.0.1:8188/system_stats | head -c 500

## Step 6: Run character replacement

Open the ComfyUI web UI (port 8188 via tunnel), load the Wan 2.2 Animate workflow, and set:

- **Mode:** `replace` (keeps background, swaps character)
- **Reference image:** `/kaggle/working/inputs/character.png`
- **Driving video:** `/kaggle/working/inputs/driving.mp4`
- **Prompt:** `2D cartoon character, Shadow Wizard Money Gang style, flat colors, bold outlines, no photorealism, no realistic skin`
- **Negative:** `photorealistic, 3d render, realistic skin texture, photograph`
- **Resolution:** 480p for test, 720p for final
- **Steps:** 20-30

Or use the API below to queue the workflow programmatically.

In [ ]:
# TODO: API-based workflow submission
# Load wan22_animate_replace.json workflow template,
# inject input paths, POST to http://127.0.0.1:8188/prompt
print("Use the ComfyUI web UI for now. API automation coming in v2.")

## Step 7: Download result

In [ ]:
# KAGGLE: Find output videos (download via the file browser UI)
import glob, os
outputs = glob.glob("/kaggle/working/ComfyUI/output/*.mp4")
outputs += glob.glob("/kaggle/working/ComfyUI/output/*.mp4")
print(f"Found {len(outputs)} output videos")
for o in outputs:
    sz = os.path.getsize(o) / 1024 / 1024
    print(f"  {o} ({sz:.1f} MB)")
if outputs:
    latest = max(outputs, key=lambda f: os.path.getmtime(f))
    print(f"\nLatest: {latest}")
    print("Download it via the file browser (right-click → Download) or:")
    print(f"  shutil.copy('{latest}', '/kaggle/working/output.mp4')")

